Recover PR244's exploratory fold0 RF-DETR union-of-cached-proposals pilot.

No inference, fitting, downloads, or scientific acceptance occurs at generation.
Runtime applies the existing refiner to pre-provisioned cached masks. The five
RF floors are a historically exposed discovery grid, not independent validation
or an adoption decision. Official PQ scores each annotator separately and pools
all strict IoU>0.5 hits. Rescue counts all baseline-blind GT (IoU<0.1), including
images with no eligible RF proposals; rescue thresholds are strict >0.3/>0.5.
RF rescue candidates satisfy the union's refined-mask minimum area before ownership.

Independently freeze artifacts/v66_expected_contract_fold0.json before execution.
It contains schema_version=1, fold=0, protocol_sha256, config_sha256,
runtime_versions, annotations_sha256, folds_sha256, preprocessed_sha256,
refiner_sha256, yolo_manifest_sha256, rfdetr_manifest_sha256, yolo_cache_sha256,
rfdetr_cache_sha256, baseline={pq,tolerance}. Hash dictionaries use UTF-8
json.dumps(mapping, sort_keys=True). No canonical fingerprints or tolerances
are invented. Each cache directory needs a separately supplied producer.json:
schema_version=1, fold=0, producer_protocol_sha256, checkpoint_sha256,
inference_floor, training_folds=[1,2], prediction_unit="filename",
preprocessing_sha256, frame=[2048,2048], files={filename:npz_sha256}.
These are supplier provenance claims; their truth/OOF independence and the real
Colab runtime remain unverified. Cache membership must exactly cover fold0.
The original YOLO and RF producers use different preprocessing (q97/q96).
Their fingerprints are separate from the refiner's consumed image fingerprint;
coordinate registration/common-observation compatibility needs external acceptance.

Preinstall numpy,pandas,scipy,torch,opencv-python,pycocotools. Linux local-only
publication is checked before staging and again at completion. No resume reuse.
Download the completed local manifest before ending the runtime. Local capacity,
copy time, GPU execution, and independent official acceptance are unverified.


In [ ]:
# Pre-provision dependencies/artifacts; no installation or inference here.
import hashlib, json, math, shutil, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm
drive.mount("/content/drive")

class CFG:
    V30 = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = V30 / "competition_data"
    ARTIFACT_ROOT = V30 / "artifacts"
    PREPROC_CACHE = V30 / "preprocessed_v30_q96"
    YOLO_OOF = Path("/content/drive/MyDrive/solar_filament_v34/prediction_cache/oof/yolo_fold_0")
    RFDETR_OOF = Path("/content/drive/MyDrive/solar_filament_v35/v45_rfdetr_oof_cache_fold0")
    LOCAL_ROOT = Path("/content/solar_v66_union")
    FOLD = 0
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    YOLO_CONF = .26
    RFDETR_GRID = (.05, .10, .20, .30, .40)
    REFINER_THRESHOLD = .45
    DROP_THRESHOLD = .90
    MIN_AREA = 20
    MANIFEST = LOCAL_ROOT / "v66_rfdetr_union_fold0_manifest.json"
    EXPECTED_CONTRACT = ARTIFACT_ROOT / "v66_expected_contract_fold0.json"

CFG.PROTOCOL_SHA256 = '2274163a70def08e68f05e96d78b69a6049b3ba747eac19546dd4c90773b3e0d'
CONFIG_KEYS = ('CROP_SIZE', 'CROP_CONTEXT', 'YOLO_CONF', 'RFDETR_GRID', 'REFINER_THRESHOLD', 'DROP_THRESHOLD', 'MIN_AREA')
def file_sha256(path):
    import hashlib
    from pathlib import Path
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def verified_checkpoint_snapshot(source, expected_sha256, destination):
    # A unique local copy isolates the loader from subsequent Drive syncs.
    # Verify the bytes actually copied, including any replacement during copy.
    import shutil
    import tempfile
    from pathlib import Path
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination, suffix=".pt", delete=False) as handle:
        local = Path(handle.name)
    try:
        shutil.copyfile(source, local)
        if file_sha256(local) != expected_sha256:
            raise ValueError("Copied checkpoint differs from frozen input identity")
        return local
    except BaseException:
        local.unlink(missing_ok=True)
        raise


def validate_fold_rows(rows, files, fold):
    from pathlib import PurePosixPath
    if type(fold) is not int or fold not in (0, 1, 2):
        raise ValueError("Invalid evaluation fold")
    mapping = {}
    for row in rows:
        name, value = row.get("file_name"), row.get("fold")
        if (not isinstance(name, str) or not name or "\\" in name
                or PurePosixPath(name).name != name or ":" in name or name in (".", "..")
                or type(value) is not int or value not in (0, 1, 2) or name in mapping):
            raise ValueError("Invalid or duplicate fold assignment")
        mapping[name] = value
    if set(mapping) != set(files):
        raise ValueError("Fold assignments must cover exactly the consumed files")
    if not any(v == fold for v in mapping.values()) or not any(v != fold for v in mapping.values()):
        raise ValueError("Training and validation partitions must be nonempty")
    return mapping


def validate_baseline(measured, expected, tolerance):
    import math
    if (any(type(v) not in (int, float) or not math.isfinite(v) for v in (measured, expected, tolerance))
            or not 0 <= measured <= 1 or not 0 <= expected <= 1 or not 0 <= tolerance < 1
            or abs(measured - expected) > tolerance):
        raise ValueError("Baseline reproduction guard failed; no result accepted")


def require_expected_contract(path, actual):
    import json
    import re
    from pathlib import Path
    expected = json.loads(Path(path).read_text(encoding="utf-8"))
    keys = {"schema_version", "fold", "protocol_sha256", "detector_sha256", "refiner_sha256",
            "annotations_sha256", "folds_sha256", "preprocessed_sha256", "baseline"}
    if not isinstance(expected, dict) or set(expected) != keys:
        raise ValueError("Expected frozen input contract is incomplete")
    baseline = expected["baseline"]
    if not isinstance(baseline, dict) or set(baseline) != {"pq", "tolerance"}:
        raise ValueError("Explicit baseline reproduction guard is required")
    validate_baseline(baseline["pq"], baseline["pq"], baseline["tolerance"])
    if type(expected["schema_version"]) is not int or type(expected["fold"]) is not int:
        raise ValueError("Invalid contract schema/fold")
    for key in keys - {"schema_version", "fold", "baseline"}:
        if not isinstance(expected[key], str) or re.fullmatch("[0-9a-f]{64}", expected[key]) is None:
            raise ValueError("Invalid expected SHA256")
    if {k:v for k,v in expected.items() if k != "baseline"} != actual:
        raise ValueError("Frozen input contract differs; do not infer")
    return expected


def validate_existing_manifest(path, expected):
    import json
    from pathlib import Path
    path = Path(path)
    if path.exists():
        manifest = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(manifest, dict) or manifest.get("input_contract") != expected:
            raise ValueError("Existing manifest contract differs; preserve it and choose a new path")


def atomic_json(path, payload):
    import json
    import os
    import tempfile
    from pathlib import Path
    path = Path(path)
    data = json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8")
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, suffix=".tmp", delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def require_local_publication(path, mountinfo=None):
    import re
    import sys
    from pathlib import Path
    path = Path(path).resolve()
    if mountinfo is None:
        if sys.platform == "win32":
            if path.drive.startswith("\\\\"):
                raise ValueError("Manifest publication requires a local filesystem")
            return  # CPU-test support only; mapped Windows drives are not classified.
        mountinfo = Path("/proc/self/mountinfo").read_text(encoding="utf-8")
    matches = []
    for line in mountinfo.splitlines():
        before, separator, after = line.partition(" - ")
        fields = before.split()
        if not separator or len(fields) < 5 or not after.split():
            continue
        point = Path(re.sub(r"\\([0-7]{3})", lambda m: chr(int(m[1], 8)), fields[4])).resolve()
        if point == path or point in path.parents:
            matches.append((len(point.parts), after.split()[0]))
    deepest = max((depth for depth, _ in matches), default=-1)
    candidates = [kind for depth, kind in matches if depth == deepest]
    if len(candidates) != 1 or candidates[0] not in {
            "overlay", "ext2", "ext3", "ext4", "xfs", "btrfs", "tmpfs", "ramfs", "rootfs", "zfs"}:
        raise ValueError("Manifest publication requires a verified local filesystem")


def publish_manifest(path, payload):
    from pathlib import Path
    path = Path(path)
    require_local_publication(path)
    reservation = path.with_name(path.name + ".publish-lock")
    try:
        reservation.mkdir()
    except FileExistsError as error:
        raise ValueError("Manifest publication is reserved; choose a new output path") from error
    try:
        if path.exists():
            raise ValueError("Manifest already exists; preserve it and choose a new output path")
        atomic_json(path, payload)
    finally:
        reservation.rmdir()



def digest_mapping(mapping):
    import hashlib
    import json
    return hashlib.sha256(json.dumps(mapping, sort_keys=True).encode()).hexdigest()

def validate_names(names):
    from pathlib import PurePosixPath
    if (not names or len(set(names)) != len(names)
            or any(not isinstance(n, str) or not n or "\\" in n or ":" in n
                or PurePosixPath(n).name != n or n in (".", "..") for n in names)
            or len({PurePosixPath(n).stem for n in names}) != len(names)):
        raise ValueError("Invalid, duplicate or colliding cache filenames")

def validate_cache_coverage(directory, names):
    from pathlib import Path
    validate_names(names)
    directory = Path(directory)
    expected = {Path(name).stem + ".npz" for name in names}
    actual = {path.name for path in directory.iterdir() if path.suffix == ".npz"}
    if actual != expected:
        raise ValueError(f"Cache coverage differs: missing={sorted(expected-actual)}, unexpected={sorted(actual-expected)}")
    paths = {name: directory / (Path(name).stem + ".npz") for name in names}
    if any(not p.is_file() or p.is_symlink() for p in paths.values()):
        raise ValueError("Cache entries must be regular files")
    return paths

def load_npz_masks(data, expected_shape):
    from io import BytesIO
    import numpy as np
    with np.load(BytesIO(data), allow_pickle=False) as archive:
        if set(archive.files) != {"packed", "shape", "confidences"}:
            raise ValueError("Cache keys differ")
        packed, shape, confs = (archive[k] for k in ("packed", "shape", "confidences"))
    if (shape.shape != (2,) or shape.dtype.kind not in "iu" or (shape <= 0).any()
            or tuple(shape.tolist()) != tuple(expected_shape)):
        raise ValueError("Cache image shape differs")
    h, w = (int(v) for v in shape)
    if (packed.dtype != np.uint8 or packed.ndim != 3 or confs.ndim != 1
            or confs.dtype.kind not in "fiu" or len(confs) != len(packed)
            or not np.isfinite(confs).all() or ((confs < 0) | (confs > 1)).any()):
        raise ValueError("Cache mask/confidence schema differs")
    # Both producers use (0,0,0) for a legitimate no-detection cache.
    if len(packed) == 0 and packed.shape == (0, 0, 0):
        return [], confs.astype(np.float32)
    if packed.shape[1:] != (h, (w + 7) // 8):
        raise ValueError("Cache packed shape differs")
    bits = np.unpackbits(packed, axis=2)
    if bits[:, :, w:].any():
        raise ValueError("Cache padding bits are not zero")
    return [mask != 0 for mask in bits[:, :, :w]], confs.astype(np.float32)

def validate_cache_manifest(manifest, names, cache_hashes, required_floor):
    import math
    import re
    validate_names(names)
    keys = {"schema_version", "fold", "producer_protocol_sha256", "checkpoint_sha256",
            "inference_floor", "training_folds", "prediction_unit", "preprocessing_sha256", "frame", "files"}
    if not isinstance(manifest, dict) or set(manifest) != keys:
        raise ValueError("Incomplete cache producer manifest")
    if (type(manifest["schema_version"]) is not int or manifest["schema_version"] != 1
            or type(manifest["fold"]) is not int or manifest["fold"] != 0
            or manifest["training_folds"] != [1, 2]
            or any(type(f) is not int for f in manifest["training_folds"])
            or manifest["frame"] != [2048, 2048]
            or any(type(v) is not int for v in manifest["frame"])
            or manifest["prediction_unit"] != "filename"):
        raise ValueError("Cache producer split/frame claims differ")
    floor = manifest["inference_floor"]
    if type(floor) not in (int, float) or not math.isfinite(floor) or not 0 <= floor <= required_floor:
        raise ValueError("Cache producer inference floor differs")
    files = manifest["files"]
    if not isinstance(files, dict) or set(files) != set(names) or files != cache_hashes:
        raise ValueError("Cache producer file identities differ")
    hashes = [manifest[k] for k in ("producer_protocol_sha256", "checkpoint_sha256", "preprocessing_sha256")]
    hashes += list(files.values())
    if any(not isinstance(v, str) or re.fullmatch("[0-9a-f]{64}", v) is None for v in hashes):
        raise ValueError("Invalid cache SHA256 identity")
    return manifest

def require_union_contract(path, actual):
    import json
    import re
    from pathlib import Path
    expected = json.loads(Path(path).read_bytes())
    if not isinstance(expected, dict) or set(expected) != set(actual) | {"baseline"}:
        raise ValueError("Incomplete independently frozen union contract")
    baseline = expected["baseline"]
    if not isinstance(baseline, dict) or set(baseline) != {"pq", "tolerance"}:
        raise ValueError("Explicit frozen baseline required")
    validate_baseline(baseline["pq"], baseline["pq"], baseline["tolerance"])
    if (type(expected["schema_version"]) is not int or expected["schema_version"] != 1
            or type(expected["fold"]) is not int or expected["fold"] != 0):
        raise ValueError("Invalid union schema/fold")
    for key in actual:
        if key.endswith("sha256") and (not isinstance(expected[key], str)
                or re.fullmatch("[0-9a-f]{64}", expected[key]) is None):
            raise ValueError("Invalid union SHA256 identity")
    if {k: v for k, v in expected.items() if k != "baseline"} != actual:
        raise ValueError("Frozen union inputs differ; do not load refiner")
    return expected

def rescue_counts(base_best, rf_matrices):
    import numpy as np
    if len(base_best) != len(rf_matrices):
        raise ValueError("Rescue annotator coverage differs")
    result = {"n_blind": 0, "rescued_03": 0, "rescued_05": 0}
    for best, matrix in zip(base_best, rf_matrices):
        best, matrix = np.asarray(best), np.asarray(matrix)
        if (best.ndim != 1 or matrix.ndim != 2 or matrix.shape[0] != len(best)
                or not np.isfinite(best).all() or not np.isfinite(matrix).all()
                or ((best < 0) | (best > 1)).any() or ((matrix < 0) | (matrix > 1)).any()):
            raise ValueError("Invalid rescue IoU arrays")
        blind = best < .1
        result["n_blind"] += int(blind.sum())
        rescued = matrix.max(axis=1) if matrix.shape[1] else np.zeros(len(best))
        result["rescued_03"] += int((blind & (rescued > .3)).sum())
        result["rescued_05"] += int((blind & (rescued > .5)).sum())
    return result


In [ ]:
require_local_publication(CFG.MANIFEST)
RUNTIME_VERSIONS = {"python": sys.version.split()[0], **{p: version(p) for p in (
    "numpy", "pandas", "scipy", "torch", "opencv-python", "pycocotools")}}
if CFG.MANIFEST.exists():
    raise ValueError("Manifest already exists; preserve it and choose a new path")
annotation_paths = list(CFG.DATA_CACHE.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
if len(annotation_paths) != 1:
    raise ValueError("Exactly one annotation JSON required")
annotation_bytes = annotation_paths[0].read_bytes()
coco = json.loads(annotation_bytes)
images = {item["id"]: item for item in coco["images"]}
if len(images) != len(coco["images"]):
    raise ValueError("Duplicate annotation image IDs")
annotations_by_image = defaultdict(list)
for ann in coco["annotations"]:
    if ann["image_id"] not in images:
        raise ValueError("Unknown annotation image ID")
    annotations_by_image[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
fold_bytes = (CFG.ARTIFACT_ROOT / "grouped_folds.csv").read_bytes()
fold_of_file = validate_fold_rows(pd.read_csv(BytesIO(fold_bytes)).to_dict("records"), sorted(file_to_ids), 0)
valid_files = sorted(name for name, fold in fold_of_file.items() if fold == 0)
validate_names(valid_files)
if any((images[i]["height"], images[i]["width"]) != (2048, 2048)
       for name in valid_files for i in file_to_ids[name]):
    raise ValueError("Annotation frame differs")
PREPROC_HASHES = {name: file_sha256(CFG.PREPROC_CACHE / name) for name in valid_files}
preprocessing_sha256 = digest_mapping(PREPROC_HASHES)
CACHE_PATHS, CACHE_HASHES, CACHE_MANIFESTS, manifest_hashes = {}, {}, {}, {}
for kind, directory, floor in (("yolo", CFG.YOLO_OOF, CFG.YOLO_CONF),
                                ("rfdetr", CFG.RFDETR_OOF, min(CFG.RFDETR_GRID))):
    paths = validate_cache_coverage(directory, valid_files)
    CACHE_PATHS[kind] = paths
    hashes = {}
    for name in tqdm(valid_files, desc=f"Validate {kind} cache"):
        captured = paths[name].read_bytes()
        hashes[name] = hashlib.sha256(captured).hexdigest()
        load_npz_masks(captured, (2048, 2048))
    producer_bytes = (directory / "producer.json").read_bytes()
    CACHE_MANIFESTS[kind] = validate_cache_manifest(json.loads(producer_bytes), valid_files,
        hashes, floor)
    CACHE_HASHES[kind] = hashes
    manifest_hashes[kind] = hashlib.sha256(producer_bytes).hexdigest()
ACTUAL_INPUTS = {"schema_version": 1, "fold": 0, "protocol_sha256": CFG.PROTOCOL_SHA256,
    "config_sha256": digest_mapping({key: getattr(CFG, key) for key in CONFIG_KEYS}),
    "runtime_versions": RUNTIME_VERSIONS, "annotations_sha256": hashlib.sha256(annotation_bytes).hexdigest(),
    "folds_sha256": hashlib.sha256(fold_bytes).hexdigest(), "preprocessed_sha256": preprocessing_sha256,
    "refiner_sha256": file_sha256(CFG.ARTIFACT_ROOT / "mediseg_refiner_fold0_resume.pt"),
    "yolo_manifest_sha256": manifest_hashes["yolo"], "rfdetr_manifest_sha256": manifest_hashes["rfdetr"],
    "yolo_cache_sha256": digest_mapping(CACHE_HASHES["yolo"]),
    "rfdetr_cache_sha256": digest_mapping(CACHE_HASHES["rfdetr"])}
INPUT_CONTRACT = require_union_contract(CFG.EXPECTED_CONTRACT, ACTUAL_INPUTS)
CFG.LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise ValueError("Pre-provision a CUDA runtime for refiner execution")
started, started_utc = time.time(), datetime.now(timezone.utc).isoformat()
torch.cuda.reset_peak_memory_stats()
print({"input_contract": "verified before refiner loading", "n_files": len(valid_files)})


In [ ]:
# [3/6] MediSegUNet(solar_filament/models/mediseg_models.py と同一)+ refinerロード
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

refiner = MediSegUNet(3, 1, False).cuda().eval()
_verified_refiner = verified_checkpoint_snapshot(
    CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
    INPUT_CONTRACT["refiner_sha256"], CFG.LOCAL_ROOT / "verified_checkpoints")
_ckpt = torch.load(_verified_refiner,
                    map_location="cuda", weights_only=False)
refiner.load_state_dict(_ckpt.get("best_model") or _ckpt["model"], strict=True)
del _ckpt


In [ ]:
# [4/6] 推論・後処理・採点関数(build_v61系と同一経路+refiner内部統計の取得)
from pycocotools import mask as mask_utils

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return output, confidences

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposal_stats(image, proposal):
    """確定構成のshrink maskと、verifier特徴用のrefiner prob統計を返す。"""
    x0, y0, x1, y1 = mask_square_box(proposal, context=CFG.CROP_CONTEXT)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    prob_full = np.zeros(proposal.shape, dtype=np.float32)
    prob_full[y0:y1, x0:x1] = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = (prob_full >= CFG.REFINER_THRESHOLD) & proposal
    inside = prob_full[proposal]
    probmass = float(inside.mean()) if inside.size else 0.0
    return refined, probmass

def encode_one(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

def decode_one(rle):
    return mask_utils.decode(rle) != 0

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    if not masks:
        return []
    occupied = np.zeros(masks[0].shape, dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

def mask_iou(a, b):
    inter = float((a & b).sum())
    union = float((a | b).sum())
    return inter / union if union > 0 else 0.0

class PanopticQuality:
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.tp_ious = []
        self.fp_count = 0
        self.fn_count = 0
    def add_matrix(self, matrix, n_gt, n_pred):
        if n_gt == 0:
            self.fp_count += n_pred; return
        if n_pred == 0:
            self.fn_count += n_gt; return
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        self.tp_ious.extend(scores)
        self.fp_count += int((hits.sum(axis=0) == 0).sum())
        self.fn_count += int((hits.sum(axis=1) == 0).sum())
    @property
    def denominator(self):
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count
    @property
    def score(self):
        d = self.denominator
        return float(sum(self.tp_ious) / d) if d > 0 else 0.0
    def sq_rq(self):
        tp = len(self.tp_ious)
        sq = float(sum(self.tp_ious) / tp) if tp > 0 else 0.0
        rq = float(tp / (tp + 0.5 * self.fp_count + 0.5 * self.fn_count)) if (tp + self.fp_count + self.fn_count) > 0 else 0.0
        return sq, rq
    def summary(self):
        sq, rq = self.sq_rq()
        return {"pq": self.score, "sq": sq, "rq": rq, "tp": len(self.tp_ious),
                "fp": self.fp_count, "fn": self.fn_count, "denominator": self.denominator}

gt_rles = {}
for image_id, info in images.items():
    height, width = int(info["height"]), int(info["width"])
    gt_rles[image_id] = [to_rle(ann, height, width) for ann in annotations_by_image[image_id] if not ann.get("iscrowd", 0)]

print("後処理・採点関数の定義OK")


In [ ]:
def score_preds(prediction, name, meter):
    best = []
    for image_id in file_to_ids[name]:
        truth = gt_rles[image_id]
        matrix = iou_matrix(truth, prediction)
        meter.add_matrix(matrix, len(truth), len(prediction))
        best.append(matrix.max(axis=1) if matrix.shape[1] else np.zeros(len(truth)))
    return best

meters = {"baseline": PanopticQuality(), **{f"union@{f}": PanopticQuality() for f in CFG.RFDETR_GRID}}
rescue = {"n_blind": 0, "rescued_03": 0, "rescued_05": 0}
for name in tqdm(valid_files, desc="Paired cached union evaluation"):
    captured = (CFG.PREPROC_CACHE / name).read_bytes()
    if hashlib.sha256(captured).hexdigest() != PREPROC_HASHES[name]:
        raise ValueError("Preprocessed bytes changed")
    image = cv2.imdecode(np.frombuffer(captured, np.uint8), cv2.IMREAD_COLOR)
    if image is None or image.shape != (2048, 2048, 3):
        raise ValueError("Preprocessed image frame differs")
    refined, confidences = {}, {}
    for kind, floor in (("yolo", CFG.YOLO_CONF), ("rfdetr", min(CFG.RFDETR_GRID))):
        cache_bytes = CACHE_PATHS[kind][name].read_bytes()
        if hashlib.sha256(cache_bytes).hexdigest() != CACHE_HASHES[kind][name]:
            raise ValueError("Cache bytes changed after contract verification")
        masks, scores = load_npz_masks(cache_bytes, (2048, 2048))
        selected = [(m, c) for m, c in zip(masks, scores) if c >= floor and m.sum() >= CFG.MIN_AREA]
        refined[kind] = [refine_proposal_stats(image, m)[0] for m, _ in selected]
        confidences[kind] = np.asarray([c for _, c in selected])
    base_masks = [m for m in refined["yolo"] if m.sum() >= CFG.MIN_AREA]
    base_prediction = [encode_one(m) for m in apply_ownership(base_masks, np.ones(len(base_masks)))]
    base_best = score_preds(base_prediction, name, meters["baseline"])
    for floor in CFG.RFDETR_GRID:
        masks = refined["yolo"] + [m for m, c in zip(refined["rfdetr"], confidences["rfdetr"]) if c >= floor]
        masks = [m for m in masks if m.sum() >= CFG.MIN_AREA]
        prediction = [encode_one(m) for m in apply_ownership(masks, np.ones(len(masks)))]
        score_preds(prediction, name, meters[f"union@{floor}"])
    rf_prediction = [encode_one(m) for m, c in zip(refined["rfdetr"], confidences["rfdetr"])
                     if c >= .10 and m.sum() >= CFG.MIN_AREA]
    matrices = [iou_matrix(gt_rles[i], rf_prediction) for i in file_to_ids[name]]
    counts = rescue_counts(base_best, matrices)
    for key in rescue:
        rescue[key] += counts[key]
out = {key: meter.summary() for key, meter in meters.items()}
baseline = out["baseline"]
validate_baseline(baseline["pq"], INPUT_CONTRACT["baseline"]["pq"], INPUT_CONTRACT["baseline"]["tolerance"])
arms = [{"rfdetr_floor": floor, **out[f"union@{floor}"],
         "delta_pq": out[f"union@{floor}"]["pq"] - baseline["pq"]} for floor in CFG.RFDETR_GRID]
manifest = {"schema_version": 1, "kind": "exploratory_v66_union_complete",
    "scientific_acceptance": "unverified supplier claims; historically exposed discovery grid; independent confirmation required",
    "input_contract": INPUT_CONTRACT, "supplier_claims": CACHE_MANIFESTS,
    "filenames": valid_files, "baseline": baseline, "arms": arms, "rescue": rescue,
    "runtime": {"service": "Google Colab", "gpu": torch.cuda.get_device_name(0),
        "started_at_utc": started_utc, "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "scope": "refiner loading and paired cache evaluation; excludes input validation/staging",
        "elapsed_seconds": time.time() - started, "gpu_peak_vram_bytes": int(torch.cuda.max_memory_allocated()),
        "compute_units_used": None, "compute_units_note": "Record separately from Colab UI; runtime API unavailable"}}
publish_manifest(CFG.MANIFEST, manifest)
print(json.dumps({"baseline": baseline, "arms": arms, "manifest": str(CFG.MANIFEST)}))


Download the completed local manifest before ending the runtime. No automatic adoption decision or resume reuse. If interrupted, preserve any existing result/reservation and choose a new MANIFEST path; do not automatically remove a potentially active reservation.